# 1. Model Creation & Providers

A comprehensive guide to initializing and using LLM models with LangChain:
- **Local Models**: Initializing local Ollama models (`ChatOllama`)
- **Direct Invocation**: Synchronous generation and inspecting output content
- **Streaming**: Streaming output tokens in real-time
- **Batch Processing**: Running concurrent inferences across multiple inputs
- **Concurrency Control**: Managing batch execution with concurrency limits
- **Universal Interface**: Dynamic provider switching via `init_chat_model`
- **Cloud Providers**: Connecting directly to cloud providers like Google GenAI

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # reads OLLAMA_MODEL from .env (set per machine)
from langchain_ollama import ChatOllama

# Initialize local Ollama model
model = ChatOllama(model=os.getenv("OLLAMA_MODEL", "qwen3:8b"), temperature=0, reasoning=False)

### 1. Local Model Initialization (`ChatOllama`)

- **Environment Loading**: Reads environment configuration via `load_dotenv()`.
- **Local Model Setup**: Connects to the local Ollama instance running the model named by `OLLAMA_MODEL` in `.env`.
- **Determinism**: Configures `temperature=0` to ensure stable, reproducible completions.

In [ ]:
resp = model.invoke("Why do dogs bark?")
print(resp.content)

### 2. Synchronous Invocation & Content Inspection

- **Invocation**: Uses `model.invoke()` for standard synchronous request-response cycles.
- **Content Extraction**: Accesses the model's textual response directly via `.content` (or `.text`).

In [ ]:
for chunk in model.stream("On how many devices can we implement offline LLMs with chatbots?"):
    print(chunk.text, end="|", flush=True)

### 3. Streaming Response Tokens

- **Real-Time Streaming**: Uses `model.stream()` to output token chunks incrementally as they are generated.
- **Latency Optimization**: Significantly reduces time-to-first-token (TTFT) for interactive and conversational applications.

In [ ]:
queries = [
    "On how many devices can we implement offline LLMs?",
    "What are the primary hardware limitations?",
    "Can we implement custom chatbots everywhere?"
]

responses = model.batch(queries)

for resp in responses:
    print(resp.text)
    print("-" * 40)

### 4. Concurrent Batch Processing

- **Batch Execution**: Uses `model.batch()` to evaluate multiple independent prompts concurrently.
- **Ordered Returns**: Returns an ordered list of response objects matching the input queries.

In [ ]:
batch_responses = model.batch(
    queries,
    config={"max_concurrency": 5}
)

print(f"Processed {len(batch_responses)} queries successfully with max_concurrency=5.")

### 5. Concurrency Control in Batch Inferences

- **Concurrency Limits**: Configures `config={"max_concurrency": 5}` to control parallel request volume.
- **Resource Protection**: Prevents local memory bottlenecks and avoids cloud API rate limit throttling.

In [ ]:
# from langchain.chat_models import init_chat_model
# chat_model = init_chat_model("google_genai:gemini-2.5-flash")
# resp = chat_model.invoke("Why do dogs bark?")
# print(resp.content)

### 6. Universal Model Initialization (`init_chat_model`)

- **Generic Factory**: Uses `init_chat_model` to instantiate chat models dynamically across different providers.
- **Portability**: Allows swapping model backends without changing downstream pipeline logic.

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI
# google_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
# resp = google_model.invoke("Why do dogs bark?")
# print(resp.content)

### 7. Direct Cloud Provider Integration (`ChatGoogleGenerativeAI`)

- **Direct Integration**: Instantiates provider-specific SDK classes such as `ChatGoogleGenerativeAI`.
- **Specialized Features**: Useful when leveraging vendor-specific capabilities, safety settings, or customized generation parameters.